In [ ]:
"""
This script performs exploratory data analysis (EDA) on a large JSON file containing hourly atmospheric data.
It processes the data in batches to avoid overwhelming RAM, especially during correlation and plotting steps.

Key steps include:
- Converting timestamps and extracting variable-wise mean values over time
- Plotting time series, correlation matrices, and variable distributions
- Detecting seasonal trends and outliers
- Computing mutual information between variables
- Performing dimensionality reduction and clustering for pattern discovery

The input JSON file must have one record per line, with fields for 'Variable', 'Time' (as ms timestamp), and 'Value' (flattened grid).
"""

import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
import seaborn as sns
from datetime import datetime
from collections import defaultdict
from sklearn.feature_selection import mutual_info_regression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore

input_file = 'FINAL_CLEANED_HOURLYatmosphericdata2021to2023.json'

#main function for processing the json and generating all plots
def eda_large_json(input_file, batch_size=10000, max_correlation_samples=50000):
    correlation_data = defaultdict(list)  #store raw values for correlation analysis
    time_series_data = defaultdict(list)  #store mean value per timestamp for time series
    variable_names = set()
    timestamps = []  #collect all timestamps for seasonal grouping
    total_records = 0

    #open and iterate through each line in the json file
    with open(input_file, 'r') as f:
        for line in tqdm(f, desc="Processing JSON lines", unit="lines"):
            record = json.loads(line.strip())
            variable = record['Variable']
            timestamp_ms = record['Time']
            values = np.array(record['Value'], dtype=np.float32)

            #convert timestamp from ms to datetime
            timestamp = datetime.utcfromtimestamp(timestamp_ms / 1000)
            timestamps.append(timestamp)

            variable_names.add(variable)

            #append the timestamp and mean to time series data
            time_series_data[variable].append((timestamp, np.mean(values)))

            #extend full list of values for correlation sampling
            correlation_data[variable].extend(values)

            total_records += 1

            #every batch_size records, show a preview time series
            if total_records % batch_size == 0:
                print(f"\nProcessed {total_records} records...")
                plot_incremental_time_series(time_series_data, total_records)

    #sample down the correlation values to avoid memory issues
    correlation_df = sample_correlation_data(correlation_data, max_samples=max_correlation_samples)

    #plot all summary analyses
    plot_correlation_matrix(correlation_df)
    plot_distributions(correlation_df)
    plot_seasonality(timestamps)
    plot_mutual_info(correlation_df)
    plot_kmeans_clustering(correlation_df)
    plot_anomalies(time_series_data)

#sample large value arrays to reduce memory footprint
def sample_correlation_data(correlation_data, max_samples=50000):
    sampled_data = {}
    for var, values in correlation_data.items():
        sampled_data[var] = np.random.choice(values, size=min(len(values), max_samples), replace=False)
    return pd.DataFrame(sampled_data)

#plot the correlation matrix between sampled variables
def plot_correlation_matrix(correlation_df):
    plt.figure(figsize=(10, 8))
    corr_matrix = correlation_df.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title("Correlation Matrix of Variables (Sampled)")
    plt.show()

#plot the distribution histogram for each variable
def plot_distributions(correlation_df):
    for variable in correlation_df.columns:
        plt.figure(figsize=(12, 6))
        sns.histplot(correlation_df[variable], kde=True, bins=50)
        plt.title(f'distribution of values for {variable}')
        plt.xlabel('value')
        plt.ylabel('frequency')
        plt.show()

#plot a rolling time series of the last 1000 samples per variable
def plot_incremental_time_series(time_series_data, total_records):
    for variable, data in time_series_data.items():
        times, values = zip(*data[-1000:])
        plt.figure(figsize=(12, 6))
        plt.plot(times, values, label=f'{variable} mean')
        plt.xlabel('time')
        plt.ylabel('mean value')
        plt.title(f'time series of {variable} (last {len(times)} records)')
        plt.legend()
        plt.show()

#plot monthly and hourly seasonal distributions
def plot_seasonality(timestamps):
    df = pd.DataFrame({'Timestamp': timestamps})
    df['Year'] = df['Timestamp'].dt.year
    df['Month'] = df['Timestamp'].dt.month
    df['Hour'] = df['Timestamp'].dt.hour

    plt.figure(figsize=(10, 6))
    sns.countplot(x='Month', data=df)
    plt.title("monthly seasonal distribution")
    plt.xlabel("month")
    plt.ylabel("frequency")
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.countplot(x='Hour', data=df)
    plt.title("hourly distribution of events")
    plt.xlabel("hour")
    plt.ylabel("frequency")
    plt.show()

#compute mutual information between all variable pairs
def plot_mutual_info(correlation_df):
    mi_scores = {}  #dictionary to hold mutual information scores for each variable pair
    variables = correlation_df.columns
    for var1 in variables:
        mi_scores[var1] = []
        for var2 in variables:
            if var1 != var2:
                #mutual_info_regression estimates how much knowing var1 reduces uncertainty about var2
                #it is a non-parametric score based on entropy estimation via k-nearest neighbors
                #higher values indicate a stronger dependency between variables (not necessarily linear)
                mi = mutual_info_regression(correlation_df[[var1]], correlation_df[var2])
                mi_scores[var1].append(mi[0])
            else:
                mi_scores[var1].append(1)  #assign maximum self-information for diagonal entries

    #convert dictionary to dataframe for plotting
    mi_matrix = pd.DataFrame(mi_scores, index=variables)
    plt.figure(figsize=(10, 8))
    sns.heatmap(mi_matrix, annot=False, cmap='Reds')
    plt.title("mutual information heatmap")
    plt.show()

#use pca and k-means to visualise clusters in 2d
def plot_kmeans_clustering(correlation_df):
    pca = PCA(n_components=2)
    data_2d = pca.fit_transform(correlation_df)

    kmeans = KMeans(n_clusters=4, random_state=42).fit(data_2d)
    plt.figure(figsize=(8, 6))
    plt.scatter(data_2d[:, 0], data_2d[:, 1], c=kmeans.labels_, cmap='viridis')
    plt.title("kmeans clustering of variables")
    plt.show()

#detect and highlight anomalies using z-score
def plot_anomalies(time_series_data):
    for variable, data in time_series_data.items():
        times, values = zip(*data)

        #z-score normalisation transforms each value as (value - mean) / std
        #this allows us to detect how many standard deviations a value is from the mean
        #absolute z-score above 2 is considered an outlier in this context
        z_scores = np.abs(zscore(values))

        #set anomalies where z-score exceeds threshold, others are marked as None
        anomalies = [v if z > 2 else None for v, z in zip(values, z_scores)]

        plt.figure(figsize=(12, 6))
        plt.plot(times, values, label=f'{variable} mean', color='blue')
        plt.scatter(times, anomalies, color='red', label='anomalies', marker='o')
        plt.xlabel('time')
        plt.ylabel('value')
        plt.title(f'anomaly detection in {variable} time series')
        plt.legend()
        plt.show()

#run the analysis
eda_large_json(input_file)
